# Assignment 2 — Fine-tuning `bert-base-multilingual-cased`

**Task:** Binary text classification — predict `is_safe` (safe vs unsafe AI-generated response)  
**Dataset:** Salamandra Guard (same splits as Assignment 1)  
**Model:** `bert-base-multilingual-cased`  
**Text input:** `response` column (raw, minimal cleaning — no lemmatisation, no stopword removal)  

Results are saved to `results/transformer_results.csv` in the same schema as `model_results.csv`.

## 0. Install dependencies
Run once, then restart the kernel.

In [1]:
# Uncomment to install
!pip install transformers datasets evaluate accelerate scikit-learn pandas numpy torch

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---

## 1. Imports

In [2]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from datasets import Dataset
import evaluate

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

PyTorch: 2.10.0+cpu
CUDA available: False
Device: cpu


## 2. Configuration

In [3]:
# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
set_seed(SEED)

# ── Paths ────────────────────────────────────────────────────────
BASE_DIR    = Path("..")
DATA_RAW    = BASE_DIR / "data" / "raw"
RESULTS_DIR = BASE_DIR / "results"
MODEL_DIR   = BASE_DIR / "models" / "bert-multilingual"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ────────────────────────────────────────────────────────
MODEL_NAME   = "bert-base-multilingual-cased"
DATASET_TAG  = "response_raw"          # label for results CSV

# ── Hyperparameters ──────────────────────────────────────────────
MAX_LENGTH   = 256
BATCH_SIZE   = 16
GRAD_ACCUM   = 2                       # effective batch = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
NUM_EPOCHS    = 3
WARMUP_RATIO  = 0.1
VAL_SIZE      = 0.1                    # 10 % of training set → validation

print(f"Model : {MODEL_NAME}")
print(f"Max length : {MAX_LENGTH}")
print(f"Batch size (per device): {BATCH_SIZE}  |  grad_accum: {GRAD_ACCUM}")
print(f"Epochs : {NUM_EPOCHS}  |  LR : {LEARNING_RATE}")

Model : bert-base-multilingual-cased
Max length : 256
Batch size (per device): 16  |  grad_accum: 2
Epochs : 3  |  LR : 2e-05


## 3. Load dataset

In [4]:
train_path = DATA_RAW / "train_raw.csv"
test_path  = DATA_RAW / "test_raw.csv"

df_train_full = pd.read_csv(train_path)
df_test       = pd.read_csv(test_path)

print(f"Train raw : {len(df_train_full):,} rows  |  columns: {list(df_train_full.columns)}")
print(f"Test  raw : {len(df_test):,} rows")
print("\nClass distribution in train (is_safe):")
print(df_train_full["is_safe"].value_counts())
print("\nClass distribution in test (is_safe):")
print(df_test["is_safe"].value_counts())

Train raw : 20,316 rows  |  columns: ['id', 'prompt', 'response', 'language', 'is_safe', 's_codes', 'majority_vote', 'majority_c_cat', 'Annotator_1', 'Annotator_2', 'Annotator_3', 'GPT_4o_LABEL_RESPONSE', 'GPT_OSS_LABEL_RESPONSE', 'Nemotron_label', 'nemo_label_og', 'prompt_length', 'response_length']
Test  raw : 1,006 rows

Class distribution in train (is_safe):
is_safe
True     10871
False     9445
Name: count, dtype: int64

Class distribution in test (is_safe):
is_safe
False    554
True     452
Name: count, dtype: int64


## 4. Minimal preprocessing

No lemmatisation, no stopword removal — transformers work on raw text.
We only fix: nulls, duplicated whitespace, leading/trailing spaces.

In [5]:
def minimal_clean(text: str) -> str:
    """Strip, collapse whitespace, remove null bytes."""
    if not isinstance(text, str):
        return ""
    text = text.replace("\x00", "")
    text = re.sub(r"[\t\r\f]+", " ", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()


def prepare_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["text"] = df["response"].apply(minimal_clean)
    # Encode label: True/1 → 1 (safe), False/0 → 0 (unsafe)
    df["label"] = df["is_safe"].map(
        lambda v: 1 if str(v).strip().lower() in {"true", "1"} else 0
    )
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"] != ""]
    return df[["text", "label", "language"]]


df_train_full = prepare_df(df_train_full)
df_test       = prepare_df(df_test)

print(f"After cleaning — train: {len(df_train_full):,}  |  test: {len(df_test):,}")
print("\nLabel distribution train:")
print(df_train_full["label"].value_counts())
print("\nLabel distribution test:")
print(df_test["label"].value_counts())

After cleaning — train: 20,310  |  test: 1,003

Label distribution train:
label
1    10865
0     9445
Name: count, dtype: int64

Label distribution test:
label
0    554
1    449
Name: count, dtype: int64


## 5. Train / Validation split

In [6]:
df_train, df_val = train_test_split(
    df_train_full,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=df_train_full["label"],
)

print(f"Train : {len(df_train):,}  |  Val : {len(df_val):,}  |  Test : {len(df_test):,}")
print(f"\nTrain label balance : {df_train['label'].mean():.3f} (fraction safe)")
print(f"Val   label balance : {df_val['label'].mean():.3f}")
print(f"Test  label balance : {df_test['label'].mean():.3f}")

Train : 18,279  |  Val : 2,031  |  Test : 1,003

Train label balance : 0.535 (fraction safe)
Val   label balance : 0.535
Test  label balance : 0.448


## 6. Tokenisation

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

# Token length stats to check if MAX_LENGTH is appropriate
sample_lengths = [
    len(tokenizer(t, truncation=False)["input_ids"])
    for t in df_train["text"].sample(500, random_state=SEED)
]
print(f"Token length (sample n=500):")
print(f"  Mean   : {np.mean(sample_lengths):.1f}")
print(f"  Median : {np.median(sample_lengths):.1f}")
print(f"  P95    : {np.percentile(sample_lengths, 95):.1f}")
print(f"  Max    : {np.max(sample_lengths)}")
print(f"  MAX_LENGTH={MAX_LENGTH} covers ~{np.mean(np.array(sample_lengths) <= MAX_LENGTH)*100:.1f}% of samples")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

C:\Users\Lenovo\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (535 > 512). Running this sequence through the model will result in indexing errors


Token length (sample n=500):
  Mean   : 216.6
  Median : 180.0
  P95    : 547.0
  Max    : 812
  MAX_LENGTH=256 covers ~73.6% of samples


## 7. Build HuggingFace Datasets

In [8]:
def df_to_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))
    ds = ds.map(tokenize_batch, batched=True, batch_size=256)
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return ds


print("Tokenising train set ...")
hf_train = df_to_hf_dataset(df_train)
print("Tokenising validation set ...")
hf_val   = df_to_hf_dataset(df_val)
print("Tokenising test set ...")
hf_test  = df_to_hf_dataset(df_test)

print(f"\nhf_train features : {hf_train.features}")

Tokenising train set ...


Map:   0%|          | 0/18279 [00:00<?, ? examples/s]

Tokenising validation set ...


Map:   0%|          | 0/2031 [00:00<?, ? examples/s]

Tokenising test set ...


Map:   0%|          | 0/1003 [00:00<?, ? examples/s]


hf_train features : {'text': Value('large_string'), 'labels': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}


## 8. Load model

In [9]:
id2label = {0: "unsafe", 1: "safe"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total params    : 177,854,978
Trainable params: 177,854,978


## 9. Metrics

In [10]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc    = accuracy_score(labels, preds)
    prec   = precision_score(labels, preds, average="macro", zero_division=0)
    rec    = recall_score(labels, preds, average="macro", zero_division=0)
    f1mac  = f1_score(labels, preds, average="macro", zero_division=0)
    f1wei  = f1_score(labels, preds, average="weighted", zero_division=0)
    return {
        "accuracy" : acc,
        "precision": prec,
        "recall"   : rec,
        "f1_macro" : f1mac,
        "f1_weighted": f1wei,
    }

## 10. Training

In [ ]:
import math

# Compute warmup_steps explicitly (replaces deprecated warmup_ratio)
steps_per_epoch = math.ceil(len(df_train) / (BATCH_SIZE * GRAD_ACCUM))
total_steps     = steps_per_epoch * NUM_EPOCHS
warmup_steps    = int(WARMUP_RATIO * total_steps)
print(f"steps_per_epoch={steps_per_epoch}  total_steps={total_steps}  warmup_steps={warmup_steps}")

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,         # replaces deprecated warmup_ratio
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    processing_class=tokenizer,        # replaces deprecated tokenizer= arg
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training ...")
train_result = trainer.train()

print("\nTraining summary:")
print(train_result.metrics)

steps_per_epoch=572  total_steps=1716  warmup_steps=171
Starting training ...


C:\Users\Lenovo\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


## 11. Evaluation on the test set

In [ ]:
# Full evaluation with HF Trainer (uses compute_metrics)
eval_output = trainer.evaluate(hf_test)
print("Test metrics (HF Trainer):")
for k, v in eval_output.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Collect raw predictions for a detailed sklearn report
preds_output = trainer.predict(hf_test)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = np.array(df_test["label"].values)

print("=" * 55)
print("Classification Report (test set)")
print("=" * 55)
print(classification_report(y_true, y_pred, target_names=["unsafe (0)", "safe (1)"]))

print("Confusion Matrix (rows=true, cols=pred):")
cm = confusion_matrix(y_true, y_pred)
print(pd.DataFrame(cm, index=["unsafe", "safe"], columns=["pred_unsafe", "pred_safe"]))

### Breakdown by language

In [ ]:
df_results = df_test.copy().reset_index(drop=True)
df_results["pred"] = y_pred

for lang, grp in df_results.groupby("language"):
    f1 = f1_score(grp["label"], grp["pred"], average="macro", zero_division=0)
    acc = accuracy_score(grp["label"], grp["pred"])
    print(f"Language={lang:4s}  n={len(grp):4d}  acc={acc:.3f}  macro-F1={f1:.3f}")

## 12. Save results

In [ ]:
acc   = accuracy_score(y_true, y_pred)
prec  = precision_score(y_true, y_pred, average="macro", zero_division=0)
rec   = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1mac = f1_score(y_true, y_pred, average="macro", zero_division=0)
f1wei = f1_score(y_true, y_pred, average="weighted", zero_division=0)

new_row = pd.DataFrame([{
    "dataset"    : DATASET_TAG,
    "model"      : MODEL_NAME,
    "accuracy"   : acc,
    "precision"  : prec,
    "recall"     : rec,
    "f1_macro"   : f1mac,
    "f1_weighted": f1wei,
}])

out_path = RESULTS_DIR / "transformer_results.csv"
if out_path.exists():
    existing = pd.read_csv(out_path)
    # Remove previous run for same model+dataset if any
    existing = existing[
        ~((existing["model"] == MODEL_NAME) & (existing["dataset"] == DATASET_TAG))
    ]
    df_out = pd.concat([existing, new_row], ignore_index=True)
else:
    df_out = new_row

df_out.to_csv(out_path, index=False)
print(f"Results saved to {out_path}")
print()
print(new_row.to_string(index=False))

## 13. Comparison with Assignment 1 baselines

In [ ]:
baseline_path = RESULTS_DIR / "model_results.csv"
df_baseline = pd.read_csv(baseline_path)

# Keep only the best classical model per dataset variant
best_classical = (
    df_baseline
    .sort_values("f1_macro", ascending=False)
    .drop_duplicates(subset=["dataset"])
    .head(5)
    [["dataset", "model", "accuracy", "precision", "recall", "f1_macro"]]
)

transformer_row = new_row[["dataset", "model", "accuracy", "precision", "recall", "f1_macro"]]

df_compare = pd.concat([best_classical, transformer_row], ignore_index=True)

print("=" * 80)
print("Comparison: Classical (best per variant) vs bert-base-multilingual-cased")
print("=" * 80)
print(df_compare.to_string(index=False))

## 14. Save fine-tuned model

In [ ]:
trainer.save_model(str(MODEL_DIR / "final"))
tokenizer.save_pretrained(str(MODEL_DIR / "final"))
print(f"Model saved to {MODEL_DIR / 'final'}")

## Summary

| Step | Details |
|---|---|
| Model | `bert-base-multilingual-cased` |
| Text input | `response` column (raw, minimal cleaning) |
| Train / Val / Test | ~18 k / ~2 k / 1 k |
| Max sequence length | 256 tokens |
| Epochs | 3 (early stopping patience=2) |
| Learning rate | 2e-5 |
| Batch size (effective) | 32 |
| Best model criterion | macro-F1 on validation |

Results saved to `results/transformer_results.csv`.  
Fine-tuned weights saved under `models/bert-multilingual/final/`.